# Scribble Evaluation Notebook
Load a scribble from a `.npy` file (Google Drive or local path), generate conditioned photos,
compute MMD vs target distribution, and check CLIP softmax gender scores.

## 1. Setup

In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "scribble_cond_loss"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/SD_cond_SD_controlnet"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. Load Scribble

In [ ]:
# ── Option A: mount Google Drive and load from there ────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# SCRIBBLE_NPY = "/content/drive/MyDrive/your_folder/final_scribble_lgd_cm.npy"

# ── Option B: download from a direct Google Drive link ──────────────────────
# File ID from: https://drive.google.com/file/d/109g8joKbyvg9uIz5QdnXIz83nkl35tR2/view
FILE_ID = "109g8joKbyvg9uIz5QdnXIz83nkl35tR2"
SCRIBBLE_NPY = f"/tmp/scribble_{FILE_ID}.npy"

if not os.path.exists(SCRIBBLE_NPY):
    import subprocess
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    subprocess.run(["gdown", url, "-O", SCRIBBLE_NPY], check=True)
    # if gdown isn't installed: !pip install -q gdown

# ── Option C: local path ────────────────────────────────────────────────────
# SCRIBBLE_NPY = "/path/to/final_scribble_lgd_cm.npy"

# ── Load ─────────────────────────────────────────────────────────────────────
arr = np.load(SCRIBBLE_NPY)            # expected shape: [N, H, W, 3] uint8
if arr.ndim == 3:                      # [H, W, 3] — wrap in batch dim
    arr = arr[np.newaxis]
print(f"Loaded: shape={arr.shape}  dtype={arr.dtype}")

scribble_pil = Image.fromarray(arr[0])  # use first image in the array
plt.figure(figsize=(4, 4))
plt.imshow(scribble_pil); plt.axis('off'); plt.title("Scribble"); plt.show()

## 3. Load Models

In [ ]:
from models    import load_models
from clip_utils import load_clip_model

architect, sprinter = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print("✅ Models loaded.")

## 4. Build Target Distribution

In [ ]:
from generation import generate_and_store_cs
from clip_utils  import encode_images_clip
from visualization import plot_row

N_TARGETS       = 6          # total target images (split evenly)
CONTROLNET_SCALE = 0.5
MAN_PROMPT   = "a superrealistic portrait photograph of a man, studio lighting"
WOMAN_PROMPT = "a superrealistic portrait photograph of a woman, studio lighting"

n_half = N_TARGETS // 2
with torch.no_grad():
    man_images,   _ = generate_and_store_cs(sprinter, MAN_PROMPT,   scribble_pil, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)
    woman_images, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, scribble_pil, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)

def pil_to_tensor(pil_list):
    return torch.cat([TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0).to(device)

with torch.no_grad():
    all_clip_embeddings = torch.cat([
        encode_images_clip(pil_to_tensor(man_images),   clip_model, clip_processor),
        encode_images_clip(pil_to_tensor(woman_images), clip_model, clip_processor),
    ], dim=0)

print(f"Target CLIP embeddings: {all_clip_embeddings.shape}")
plot_row(man_images,   f"Target Man ({n_half})",   count=n_half)
plot_row(woman_images, f"Target Woman ({n_half})", count=n_half)

## 5. Generate Eval Photos from Scribble

In [ ]:
N_EVAL       = 10
EVAL_PROMPT  = "a superrealistic professional photograph of"

sprinter.vae.to(dtype=torch.float16)
eval_photos = []
with torch.no_grad():
    for start in range(0, N_EVAL, 2):
        bs = min(2, N_EVAL - start)
        result = sprinter(
            prompt=[EVAL_PROMPT] * bs,
            image=[scribble_pil] * bs,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=CONTROLNET_SCALE,
            output_type="pil",
        )
        eval_photos.extend(result.images)
sprinter.vae.to(dtype=torch.float32)

print(f"Generated {len(eval_photos)} eval photos.")
plot_row(eval_photos, "Eval Photos from Scribble", count=min(10, len(eval_photos)))

## 6. Compute MMD vs Target

In [ ]:
from metrics import compute_mmd

with torch.no_grad():
    eval_clip = encode_images_clip(pil_to_tensor(eval_photos), clip_model, clip_processor)

mmd_val = compute_mmd(eval_clip, all_clip_embeddings).item()
print(f"MMD (eval photos vs target): {mmd_val:.6f}")

## 7. CLIP Softmax Gender Scores

In [ ]:
import torch.nn.functional as F

text_inputs = clip_processor(
    text=[MAN_PROMPT, WOMAN_PROMPT],
    return_tensors="pt", padding=True,
).to(device)

with torch.no_grad():
    text_feats = clip_model.get_text_features(**text_inputs)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

    logits = (eval_clip @ text_feats.T) * 100.0
    probs  = F.softmax(logits, dim=-1).cpu().numpy()

n_male   = (probs[:, 0] > 0.5).sum()
n_female = (probs[:, 1] > 0.5).sum()
print(f"Male: {n_male}  Female: {n_female}  (out of {len(eval_photos)})")
print(f"Mean p(male): {probs[:,0].mean():.3f}   Mean p(female): {probs[:,1].mean():.3f}")

# ── Display with gender labels ───────────────────────────────────────────────
n_cols = min(len(eval_photos), 5)
n_rows = (len(eval_photos) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
axes = np.array(axes).flatten()
for idx, (img, p) in enumerate(zip(eval_photos, probs)):
    label  = "M" if p[0] > 0.5 else "F"
    color  = "royalblue" if label == "M" else "crimson"
    axes[idx].imshow(img)
    axes[idx].set_title(f"{label}  p={max(p):.2f}", fontsize=9, color=color)
    axes[idx].axis('off')
for ax in axes[len(eval_photos):]:
    ax.axis('off')
fig.suptitle(f"Eval Photos — {n_male}M / {n_female}F  |  MMD={mmd_val:.4f}",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. CLIP PCA — Eval vs Target

In [ ]:
from sklearn.decomposition import PCA

target_np = all_clip_embeddings.cpu().numpy()
eval_np   = eval_clip.cpu().numpy()
combined  = np.vstack([target_np, eval_np])

pca    = PCA(n_components=2)
coords = pca.fit_transform(combined)

t_man   = coords[:n_half]
t_woman = coords[n_half : 2 * n_half]
e_pts   = coords[2 * n_half :]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(t_man[:,0],   t_man[:,1],   c='royalblue', s=60, alpha=0.7, label='Target (man)')
ax.scatter(t_woman[:,0], t_woman[:,1], c='crimson',   s=60, alpha=0.7, label='Target (woman)')
ax.scatter(e_pts[:,0],   e_pts[:,1],   c='limegreen', s=80, alpha=0.9, marker='x', label=f'Eval (MMD={mmd_val:.4f})')
ax.set_title(f"CLIP PCA — var={pca.explained_variance_ratio_.sum():.1%}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()